# Cleaning Data (continued): Reshaping
v.ekc

Picking up from last class — we dive deeper into reshaping: wide ↔ long transformations, handling multiple values per cell, and flattening multi-level column names.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Reshaping Data — Wide → Long |
| 3 | Multiple Values per Cell |
| 4 | 🔬 Activity — Pets DataFrame |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import pandas as pd

---
## 2. Reshaping Data — Wide → Long

The `sample_weather.csv` dataset has one column per weather variable — let's transform it into a long format where each row is one observation.

### 📋 Board Reference

| Direction | Function | When to use |
|-----------|----------|-------------|
| Wide → Long | `.melt(id_vars=, value_vars=)` | Each measurement → its own row |
| Wide → Long | `.stack()` | Column names become a new index level |
| Long → Wide | `.pivot(index=, columns=, values=)` | One value per cell — no aggregation |
| Long → Wide | `.pivot_table(aggfunc=)` | Multiple values per cell — aggregate them |

In [ ]:
df_weather_wide = pd.read_csv('sample_weather.csv')
df_weather_wide = df_weather_wide.iloc[:,1:]
df_weather_wide

In [ ]:
# transpose with more informative columns
date_index = df_weather_wide.set_index('date')
date_index

In [ ]:
# Make a long series
long_df = date_index.stack()
long_df

In [ ]:
long_df = long_df.reset_index(name='value')
long_df

In [ ]:
long_df.rename(columns = {'level_1':'variable'},inplace=True)
long_df

In [ ]:
# If our columns had a name
date_index.columns.name = 'variable'

In [ ]:
date_index

In [ ]:
# Now renaming isn't necessary
date_index.stack().reset_index(name = 'value')

In [ ]:
# another way
df_weather_wide

In [ ]:
# another way
long_df = df_weather_wide.melt(id_vars = 'date',value_vars = ['max_temp','min_temp','inches_of_rain'])
long_df

In [ ]:
# change long format back into wide format
long_df.pivot(index = 'date',columns = 'variable',values='value')

---
## 3. Multiple Values per Cell

`.pivot()` assumes exactly one value per index/column combination. If there are duplicates, it raises an error — that's when `.pivot_table()` steps in.

In [ ]:
# A new long dataframe
long_df = pd.read_csv('long_data.csv')
long_df = long_df.iloc[:,1:]
long_df.head()

In [ ]:
# check the number of entries for each combination of date/category
pd.crosstab(index=long_df.date,columns=long_df.category)

In [ ]:
# Pivot doesn't know how to handle this
long_df.pivot(index='date', columns='category', values='sales')

In [ ]:
# Use pivot table instead to get the average sales by date and category
long_df.pivot_table(index='date', columns='category', values='sales')

In [ ]:
# You can also change the aggregation function; e.g. TOTAL sales by date/category
wide_df = long_df.pivot_table(index='date', columns='category', values='sales', aggfunc=sum)
wide_df

In [ ]:
# Can also use it like crosstab if you choose len as the aggfunc
long_df.pivot_table(index=['date'], columns='category', values=['sales'], aggfunc=len)

In [ ]:
# back to a longer format (note that this only has total sales)
wide_df.reset_index().melt(id_vars='date', value_vars=['Books','Clothing','Electronics'])

In [ ]:
# You can also change choose multiple columns
wide_df2 = long_df.pivot_table(index='date', columns=['category','product'], values='sales', aggfunc=sum)
wide_df2

In [ ]:
# Rename columns and reset index to work with it as you normally would
wide_df2.columns = list(map("_".join, wide_df2.columns))
wide_df2.reset_index()

---
## 4. 🔬 Activity — Pets DataFrame

Practice the full cleaning + reshaping workflow on a small dataset.

In [ ]:
# Create a DataFrame with data cleaning and reshaping opportunities
data = {
    'Pet Name': ['Fluffy', 'Whiskers', 'Bubbles', 'Spike', 'Coco', 'Maybelle', 'Snowball'],
    'Date Adopted': ['10-01-2023','03-04-2024','01-10-2024','02-14-2024','11-22-2023','01-04-2024','12-25-2025'],
    'Animal Type': ['Cat', 'Cat', 'Fish', 'Dog', 'Fish', 'Dog', 'Cat'],
    'Pet Age': ['3', '2', '13', '5', '4', '3', '2'],
    'Color': ['White', 'Gray', 'Orange', 'White', 'White', 'Black', 'Black'],
    'Happiness Level': ['High', 'Medium', 'High', 'Low', 'High', 'High', 'Medium']
}
df_pets = pd.DataFrame(data)
df_pets

---
### Try It 1: Clean & Reshape the Pets DataFrame 😊

1. **Rename columns** to a consistent format (e.g., `snake_case` — your choice).

2. **Fix datatypes** — convert `Pet Age` to int and `Date Adopted` to datetime.

3. **Pivot** — use `.pivot_table()` to find the average age by `Animal Type`.

In [ ]:
# 1. Rename columns


In [ ]:
# 2. Fix datatypes


In [ ]:
# 3. pivot_table: average age by Animal Type


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Many column naming conventions work — consistency is what matters.*

```python
# 1. Rename
df_pets = df_pets.rename(columns={
    'Pet Name': 'pet_name',
    'Date Adopted': 'date_adopted',
    'Animal Type': 'animal_type',
    'Pet Age': 'pet_age',
    'Color': 'color',
    'Happiness Level': 'happiness_level'
})

# 2. Fix types
df_pets['pet_age'] = df_pets['pet_age'].astype(int)
df_pets['date_adopted'] = pd.to_datetime(df_pets['date_adopted'], format='%m-%d-%Y')

# 3. pivot_table
df_pets.pivot_table(index='animal_type', values='pet_age', aggfunc='mean')
```

</details>

---
## Appendix — Reshaping Quick Reference

### Wide → Long
```python
# stack
df.set_index('id').stack().reset_index(name='value')

# melt
df.melt(id_vars='id_col', value_vars=['a', 'b'])
```

### Long → Wide
```python
# pivot (one value per cell — errors if duplicates exist)
df.pivot(index='id', columns='var', values='val')

# pivot_table (aggregates duplicates)
df.pivot_table(index='id', columns='var', values='val', aggfunc=sum)

# crosstab equivalent
df.pivot_table(index='id', columns='var', values='val', aggfunc=len)
```

### Flatten multi-level columns
```python
df.columns = list(map('_'.join, df.columns))
df.reset_index()
```